# Tugas 1 Kelompok 7
1. Habil Aarrafi 2411511014
2. Najwa Azzahra 2411513004
3. Adam Maulana  2411513032
#Tugas Latihan – Sistem Embedded dan IoT



## 1. Hands-On 1 – Skenario gateway edge-AI untuk pemrosesan citra

### Skenario
Sebuah sistem kamera digunakan untuk mendeteksi objek secara langsung di lokasi (edge), tanpa harus selalu mengirim citra ke cloud. Platform yang dipilih harus mampu memproses citra, memiliki konsumsi daya yang masih dapat diterima, biaya yang sesuai, serta mudah dikembangkan.

Alternatif platform:
- **ESP32-S3:** biaya dan konsumsi daya rendah, tetapi kemampuan pemrosesan citra/AI relatif terbatas.
- **Raspberry Pi 5:** kemampuan komputasi lebih tinggi dan fleksibel untuk pengembangan.
- **NVIDIA Jetson Orin Nano:** dirancang untuk komputasi AI di edge dan cocok untuk pemrosesan citra yang lebih berat, tetapi biaya serta konsumsi dayanya lebih tinggi.

### Bobot kriteria
Bobot dibuat berbeda untuk menyesuaikan kebutuhan pemrosesan citra. Skor setiap platform menggunakan skala 1–5 (5 = paling baik). Untuk kriteria biaya dan konsumsi daya, skor tinggi berarti lebih murah dan lebih hemat daya.


In [5]:
import pandas as pd

# Bobot kriteria (total 100%)
weights = {
    "Kemampuan AI/pemrosesan citra": 0.40,
    "Konsumsi daya": 0.15,
    "Biaya": 0.15,
    "Kemudahan pengembangan": 0.15,
    "Konektivitas": 0.15
}

# Skor penilaian 1–5 (asumsi untuk skenario tugas)
scores = pd.DataFrame({
    "Kemampuan AI/pemrosesan citra": [2, 4, 5],
    "Konsumsi daya": [5, 3, 2],
    "Biaya": [5, 4, 2],
    "Kemudahan pengembangan": [3, 5, 4],
    "Konektivitas": [4, 5, 4]
}, index=["ESP32-S3", "Raspberry Pi 5", "Jetson Orin Nano"])

weighted_scores = scores.mul(pd.Series(weights)).sum(axis=1)
result = pd.DataFrame({
    "Nilai tertimbang": weighted_scores,
    "Peringkat": weighted_scores.rank(ascending=False, method="min").astype(int)
}).sort_values("Peringkat")

print("Bobot kriteria:")
display(pd.Series(weights, name="Bobot").to_frame())
print("Skor masing-masing platform:")
display(scores)
print("Hasil perhitungan:")
display(result)


Bobot kriteria:


,Bobot
Kemampuan AI/pemrosesan citra,0.40
Konsumsi daya,0.15
Biaya,0.15
Kemudahan pengembangan,0.15
Konektivitas,0.15


Skor masing-masing platform:


,Kemampuan AI/pemrosesan citra,Konsumsi daya,Biaya,Kemudahan pengembangan,Konektivitas
ESP32-S3,2,5,5,3,4
Raspberry Pi 5,4,3,4,5,5
Jetson Orin Nano,5,2,2,4,4


Hasil perhitungan:


,Nilai tertimbang,Peringkat
Raspberry Pi 5,4.15,1
Jetson Orin Nano,3.80,2
ESP32-S3,3.35,3


### Analisis hasil

Pada skenario ini, kemampuan AI/pemrosesan citra diberi bobot paling besar, yaitu 40%, karena sistem harus menganalisis gambar secara langsung. Jetson Orin Nano memperoleh skor tinggi pada kemampuan AI, tetapi nilainya berkurang karena konsumsi daya dan biaya. Raspberry Pi 5 memiliki nilai yang seimbang pada kemampuan pemrosesan, biaya, konektivitas, dan kemudahan pengembangan.

Berdasarkan bobot dan skor asumsi di atas, **Raspberry Pi 5 menjadi rekomendasi apabila hasil perhitungan menunjukkan nilai tertinggi**. Rekomendasi dapat berubah jika bobot kemampuan AI dinaikkan lebih besar atau jika kebutuhan utamanya adalah pemrosesan AI yang berat. Dalam kondisi tersebut, Jetson Orin Nano dapat menjadi pilihan yang lebih sesuai. ESP32-S3 lebih cocok untuk sistem sederhana dengan kebutuhan daya dan biaya rendah, bukan pemrosesan citra yang kompleks.

**Kesimpulan:** rekomendasi platform dapat berubah karena bobot kriteria mencerminkan prioritas aplikasi. Pemilihan tidak hanya ditentukan oleh spesifikasi tertinggi, tetapi juga kesesuaian dengan kebutuhan sistem.


## 2. Hands-On 4 – Perbandingan ADC 12-bit dan 16-bit untuk sensor suhu

### Dasar perhitungan
Resolusi ADC menunjukkan banyaknya tingkatan nilai digital yang dapat digunakan untuk merepresentasikan tegangan analog. Semakin besar jumlah bit, semakin kecil perubahan tegangan yang dapat dibedakan.

Asumsi simulasi:
- Tegangan referensi ADC = 3,3 V.
- Rentang pengukuran suhu = 0–100 °C.
- Sensor memiliki rentang keluaran 0–1,0 V untuk rentang suhu tersebut (setara 10 mV/°C).
- ADC ideal, sehingga efek noise dan ketidakakuratan sensor belum dimasukkan.

Rumus:
- Jumlah tingkat ADC = \(2^N\)
- Ukuran 1 LSB = \(V_{ref}/2^N\)
- Perubahan suhu per LSB = \(LSB / 0,01\), karena sensitivitas sensor 0,01 V/°C.


In [6]:
Vref = 3.3                 # volt
sensor_sensitivity = 0.01 # volt per derajat Celsius
temperature_range = (0, 100)

def adc_parameters(bits):
    levels = 2 ** bits
    lsb_volt = Vref / levels
    temperature_step = lsb_volt / sensor_sensitivity
    half_lsb_error = temperature_step / 2
    return {
        "Resolusi ADC": f"{bits}-bit",
        "Jumlah tingkat": levels,
        "Ukuran 1 LSB (V)": lsb_volt,
        "Ukuran 1 LSB (mV)": lsb_volt * 1000,
        "Perubahan suhu per LSB (°C)": temperature_step,
        "Batas kuantisasi ideal (±°C)": half_lsb_error
    }

adc_comparison = pd.DataFrame([
    adc_parameters(12),
    adc_parameters(16)
]).set_index("Resolusi ADC")

display(adc_comparison)
print("Peningkatan jumlah tingkat:", (2**16) / (2**12), "kali")
print("Pengecilan langkah suhu:", adc_parameters(12)["Perubahan suhu per LSB (°C)"] /
      adc_parameters(16)["Perubahan suhu per LSB (°C)"], "kali")


,Jumlah tingkat,Ukuran 1 LSB (V),Ukuran 1 LSB (mV),Perubahan suhu per LSB (°C),Batas kuantisasi ideal (±°C)
Resolusi ADC,,,,,
12-bit,4096,0.000806,0.805664,0.080566,0.040283
16-bit,65536,0.000050,0.050354,0.005035,0.002518


Peningkatan jumlah tingkat: 16.0 kali
Pengecilan langkah suhu: 16.0 kali


In [7]:
# Simulasi pembacaan tegangan sensor pada suhu tertentu
import numpy as np

temperatures = np.array([0, 10, 25, 37, 50, 75, 100], dtype=float)
sensor_voltages = temperatures * sensor_sensitivity

def adc_reading(voltage, bits):
    max_code = (2 ** bits) - 1
    code = np.round((voltage / Vref) * max_code)
    code = np.clip(code, 0, max_code)
    measured_voltage = code / max_code * Vref
    estimated_temperature = measured_voltage / sensor_sensitivity
    return code.astype(int), estimated_temperature

simulation = pd.DataFrame({"Suhu sebenarnya (°C)": temperatures,
                           "Tegangan sensor (V)": sensor_voltages})
for bits in [12, 16]:
    codes, estimates = adc_reading(sensor_voltages, bits)
    simulation[f"Kode ADC {bits}-bit"] = codes
    simulation[f"Hasil suhu {bits}-bit (°C)"] = estimates
    simulation[f"Galat {bits}-bit (°C)"] = estimates - temperatures

display(simulation.round(5))


,Suhu sebenarnya (°C),Tegangan sensor (V),Kode ADC 12-bit,Hasil suhu 12-bit (°C),Galat 12-bit (°C),Kode ADC 16-bit,Hasil suhu 16-bit (°C),Galat 16-bit (°C)
0,0.0,0.00,0,0.00000,0.00000,0,0.00000,0.00000
1,10.0,0.10,124,9.99267,-0.00733,1986,10.00046,0.00046
2,25.0,0.25,310,24.98168,-0.01832,4965,25.00114,0.00114
3,37.0,0.37,459,36.98901,-0.01099,7348,37.00069,0.00069
4,50.0,0.50,620,49.96337,-0.03663,9930,50.00229,0.00229
5,75.0,0.75,931,75.02564,0.02564,14894,74.99840,-0.00160
6,100.0,1.00,1241,100.00733,0.00733,19859,99.99954,-0.00046


### Analisis peningkatan presisi

ADC 12-bit memiliki 4.096 tingkat, sedangkan ADC 16-bit memiliki 65.536 tingkat. Dengan tegangan referensi 3,3 V, ukuran satu LSB ADC 12-bit sekitar 0,806 mV, sedangkan ADC 16-bit sekitar 0,050 mV. Berdasarkan sensitivitas sensor 10 mV/°C, perubahan suhu per LSB menjadi sekitar 0,0806 °C pada ADC 12-bit dan 0,0050 °C pada ADC 16-bit. Secara ideal, langkah pengukuran ADC 16-bit sekitar 16 kali lebih kecil.

Peningkatan ini **signifikan dari sisi resolusi atau kemampuan membedakan perubahan kecil**. Namun, peningkatan resolusi belum tentu membuat pengukuran suhu nyata 16 kali lebih akurat. Akurasi akhir juga dipengaruhi ketelitian sensor, noise, kestabilan tegangan referensi, dan kualitas rangkaian. Jika sensor memiliki galat yang jauh lebih besar daripada 0,005 °C, manfaat tambahan ADC 16-bit mungkin tidak terlihat pada hasil pengukuran. Jadi, ADC 16-bit berguna ketika sensor dan rangkaian pendukung juga cukup presisi.


## 3. Hands-On 6 – Estimasi umur baterai dengan interval pengiriman 1 menit

### Asumsi perhitungan
Karena data arus dari Hands-On 6 tidak tersedia pada soal, digunakan contoh parameter berikut:
- Kapasitas baterai = 2.000 mAh.
- Arus saat perangkat tidur/siaga = 0,1 mA.
- Arus saat mengirim data = 80 mA.
- Lama satu kali pengiriman = 2 detik.
- Interval awal = 5 menit (300 detik).
- Interval baru = 1 menit (60 detik).

Model menghitung arus rata-rata dari konsumsi saat siaga dan tambahan konsumsi ketika radio mengirim data. Estimasi ini menganggap kapasitas baterai dapat digunakan seluruhnya dan tidak memperhitungkan penurunan kapasitas, efisiensi regulator, serta perubahan kondisi baterai.


In [8]:
battery_capacity_mAh = 2000
sleep_current_mA = 0.1
transmit_current_mA = 80
transmit_duration_s = 2

def estimate_battery(interval_minutes):
    interval_s = interval_minutes * 60
    duty_cycle = transmit_duration_s / interval_s
    average_current = (sleep_current_mA * (1 - duty_cycle) +
                       transmit_current_mA * duty_cycle)
    lifetime_hours = battery_capacity_mAh / average_current
    lifetime_days = lifetime_hours / 24
    return {
        "Interval kirim (menit)": interval_minutes,
        "Arus rata-rata (mA)": average_current,
        "Estimasi umur (jam)": lifetime_hours,
        "Estimasi umur (hari)": lifetime_days
    }

battery_result = pd.DataFrame([
    estimate_battery(5),
    estimate_battery(1)
]).set_index("Interval kirim (menit)")

display(battery_result.round(2))

old_days = battery_result.loc[5, "Estimasi umur (hari)"]
new_days = battery_result.loc[1, "Estimasi umur (hari)"]
print(f"Penurunan estimasi umur baterai: {old_days - new_days:.2f} hari")
print(f"Umur baterai pada interval 1 menit menjadi {new_days/old_days*100:.1f}% dari interval 5 menit.")


,Arus rata-rata (mA),Estimasi umur (jam),Estimasi umur (hari)
Interval kirim (menit),,,
5,0.63,3161.22,131.72
1,2.76,723.76,30.16


Penurunan estimasi umur baterai: 101.56 hari
Umur baterai pada interval 1 menit menjadi 22.9% dari interval 5 menit.


### Analisis trade-off

Ketika interval pengiriman dipersingkat dari 5 menit menjadi 1 menit, data dikirim lebih sering sehingga monitoring menjadi lebih responsif. Perubahan kondisi sensor dapat diketahui lebih cepat karena jeda antar-pengiriman lebih singkat. Hal ini bermanfaat untuk sistem yang membutuhkan pemantauan hampir real-time.

Namun, frekuensi pengiriman yang meningkat membuat perangkat lebih sering mengaktifkan komunikasi nirkabel. Akibatnya, arus rata-rata bertambah dan umur baterai menjadi lebih pendek. Berdasarkan asumsi simulasi, estimasi umur baterai turun dari sekitar 5,5 bulan menjadi sekitar 1 bulan (nilai tepat ditampilkan oleh program). Dengan demikian, terdapat trade-off antara **responsivitas monitoring** dan **umur baterai**. Interval 1 menit cocok ketika data cepat sangat dibutuhkan, sedangkan interval 5 menit lebih hemat energi jika pemantauan tidak harus terlalu sering.

> Untuk menyesuaikan hasil dengan Hands-On 6 asli, ganti kapasitas baterai, arus siaga, arus pengiriman, dan durasi pengiriman pada bagian kode sesuai data percobaan.


## Kesimpulan

1. Pada pemilihan platform edge-AI, perubahan bobot kriteria dapat mengubah platform yang direkomendasikan. Platform harus dipilih berdasarkan prioritas aplikasi pemrosesan citra.
2. ADC 16-bit memiliki resolusi lebih tinggi daripada ADC 12-bit. Akan tetapi, peningkatan akurasi nyata tetap dibatasi oleh sensor, noise, dan rangkaian pendukung.
3. Interval pengiriman 1 menit membuat monitoring lebih responsif, tetapi meningkatkan konsumsi energi sehingga umur baterai lebih pendek dibandingkan interval 5 menit.
